# External review: exploratory seed check of crop + augment + layer mix

In the variant runs, crop + augment + layer mix had the lowest Tortoise (CM04) EER,
26.43%, below every seed of the published recipe (30.14-38.43%). Its dev EER was worse
(9.10% against 5.50%), so the rule fixed in advance did not select it, and the
published recipe stays.

Because this variant is singled out **after** seeing a test-set score, this run is
**exploratory** and is reported as such, never as a selection. It trains the same
variant with seeds 2025, 7, 42 and 99, so its CM04 result over five seeds can be set
against the published recipe's five seeds (35.59 +/- 3.35).

| Block | Runs | Time on a T4 |
|---|---|---|
| explore_seeds | crop + augment + layer mix, seeds 2025, 7, 42, 99 | ~45 min |

**How to run.** Inputs: `saikrishnareddy9/codemix-bundle-normalised` and
`saikrishnareddy9/codemix-s3-native-results`. Settings: GPU T4, Internet On. Click
**Run All** once; nothing needs changing.

**Getting the results.** Section 3 ends with a link: click it and
`explore_seeds_results.zip` downloads. The zip is also rewritten after every finished
run, so if the page stops updating, download it from the right-hand panel: Output ->
/kaggle/working -> explore_seeds_results.zip -> the three dots -> Download.

## 1. Settings

In [ ]:
SMOKE = False      # True -> ~20 min check of every block with tiny runs; numbers mean nothing

BLOCKS = ["explore_seeds"]
SEEDS = [2025, 7, 42, 99]            # extra seeds; the published runs used 1234
VARIANTS = {                          # opt-in options, added to the deployed recipe
    "crop": {"crop_per_epoch": True},
    "crop_aug": {"crop_per_epoch": True, "augment": "channel_v1"},
    "crop_layers": {"crop_per_epoch": True, "layer_weighting": True},
    "crop_aug_layers": {"crop_per_epoch": True, "augment": "channel_v1", "layer_weighting": True},
}
TIE = 0.005                           # dev-EER ties within half a point go to the simpler variant
#: Candidates finished in an earlier run, which the selection rule must also see. Their
#: test-set EERs come along for the report only; choosing uses dev EER alone.
PRIOR = [  # crop_aug_layers, seed 1234, from the layer-variants run (2026-10-14)
    {"name": "deployed_crop_aug_layers_s1234", "block": "variants", "system": "deployed", "seed": 1234, "options": {"crop_per_epoch": True, "augment": "channel_v1", "layer_weighting": True}, "dev_eer": 0.091, "eval_pool": 0.03379709450830141, "cm04": 0.26427997791497265, "rvc_test": 0.07270144358326196, "status": "ok", "earlier": True},
]
#: explore_seeds block: the variant (a key of VARIANTS) to repeat over SEEDS. It is
#: chosen after seeing test scores, so its results are exploratory, never a selection.
EXPLORE = "crop_aug_layers"

# ---- nothing below should need changing ------------------------------------------
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week12-krishna-paper-chapters"
BASE = {                              # system -> (published config, published result prefix)
    "deployed": ("configs/train_lora_norm_rvc_channel.yaml", "results/w10/lora_norm_rvc_channel"),
    "xtts_channel": ("configs/train_lora_norm_channel.yaml", "results/w10/lora_norm_channel"),
    "s3_xtts_channel": ("configs/train_s3_native_channel.yaml",
                        "docs/results/s3_native/s3_native_channel"),
}
EVALS = {
    "eval_pool": "data/manifests/codemix_eval_channel20.csv",
    "cm04": "data/manifests/score_cm04_norm_channel20.csv",
    "rvc_test": "data/manifests/rvc_holdout_test_channel20.csv",
}
SCORE_ARGS = "--device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0"

## 2. Setup: clone, Stage-1 checkpoint, data check, config check

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"
OUT = WORK / "review_results"


def sh(cmd, check=True):
    """Run a short command and show its output."""
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True,
                       cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def run_logged(args):
    """Run a long command, streaming its output live, and return all of it."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, text=True, errors="replace", cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:  # the full output is kept
        log.write(f"\n$ {' '.join(args)}\n")
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if not any(noise in line for noise in ("Warning", "warn(", "Some weights", "You should")):
                print(line, end="", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines)


def _make_writable(func, path, _):
    """git marks its pack files read-only; clear that so a re-run can remove them."""
    os.chmod(path, 0o700)
    func(path)


# Skip LFS on clone: a normal clone would download every checkpoint in the repo.
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(1, 4):  # GitHub occasionally drops a connection
    if REPO.exists():
        shutil.rmtree(REPO, onerror=_make_writable)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    print(f"clone attempt {attempt} failed; retrying in 20 s", flush=True)
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository after 3 attempts. Is Internet switched On?")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
shutil.rmtree(OUT, ignore_errors=True)
(WORK / "explore_seeds_results.zip").unlink(missing_ok=True)
OUT.mkdir()
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()


def save_progress():
    """Zip the results so far (no checkpoints) into /kaggle/working/explore_seeds_results.zip."""
    import zipfile
    staging = WORK / "explore_seeds_results.zip.tmp"
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix != ".pt":
                zf.write(f, f.relative_to(WORK))
    staging.replace(WORK / "explore_seeds_results.zip")


FAILURES = []


def tail(text, n=30):
    return "\n".join([line for line in text.splitlines() if line.strip()][-n:])


def failed(where, detail):
    """Record a failed step loudly, save it, and let the run continue."""
    FAILURES.append({"where": where, "detail": detail})
    print(f"\n!!!!!! FAILED: {where}\n{detail}\n!!!!!! recorded; the run continues\n", flush=True)
    try:
        (OUT / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
        save_progress()
    except Exception as e:
        print("(could not write FAILURES.txt:", e, ")")


# ---- the Stage-1 checkpoint every S2 run starts from, verified by its LFS hash ----
BASELINE = REPO / "checkpoints" / "baseline" / "best.pt"
# git stores the LFS pointer (with the file's sha256) whether or not the file was fetched
pointer = subprocess.run(["git", "cat-file", "-p", "HEAD:checkpoints/baseline/best.pt"],
                         cwd=REPO, text=True, capture_output=True).stdout
want = re.search(r"oid sha256:([0-9a-f]{64})", pointer)
if not want:
    raise SystemExit("checkpoints/baseline/best.pt is not tracked by LFS -- unexpected repo state")
want = want.group(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


source = None
for candidate in glob.glob("/kaggle/input/**/checkpoints/baseline/best.pt", recursive=True):
    if os.path.getsize(candidate) > 100_000_000 and sha256(candidate) == want:
        shutil.copy(candidate, BASELINE)
        source = candidate
        break
if source is None:
    print("Stage-1 checkpoint not found in the inputs; fetching it with git lfs ...")
    sh("git lfs install --local && git lfs pull --include checkpoints/baseline/best.pt", check=False)
    source = "git lfs"
if BASELINE.stat().st_size < 100_000_000 or sha256(BASELINE) != want:
    raise SystemExit("The Stage-1 checkpoint could not be obtained. Attach the dataset "
                     "saikrishnareddy9/codemix-s3-native-results (it contains it) and run again.")
print(f"Stage-1 checkpoint OK ({BASELINE.stat().st_size / 1e6:.0f} MB, sha256 verified) from {source}")

sh("git log --oneline -1")
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0",
   check=False)

import pandas as pd
import yaml
from src.data.scoring_manifests import missing_files

channel = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
if not channel:
    raise SystemExit("Attach the dataset saikrishnareddy9/codemix-bundle-normalised (Add Input).")
ROOT = str(Path(channel[0]).parent)
print("data root:", ROOT)


def make_config(name, base_config, seed, options=None):
    """A copy of a published config with only the seed, out_dir and opt-in options changed."""
    cfg = yaml.safe_load((REPO / base_config).read_text())
    cfg.update({"seed": int(seed), "out_dir": f"checkpoints/review/{name}",
                "wandb_mode": "disabled"})
    cfg.update(options or {})
    path = REPO / "configs" / f"review_{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return str(path.relative_to(REPO)), cfg


manifests = set(EVALS.values())
for base_config, _ in BASE.values():
    cfg = yaml.safe_load((REPO / base_config).read_text())
    manifests |= {cfg["train_manifest"], cfg["dev_manifest"]}
for manifest in sorted(manifests):
    gone = missing_files(pd.read_csv(manifest), ROOT)
    print(f"  {manifest:58s} missing {len(gone)}")
    assert not gone, f"{manifest} has clips not in the dataset: {gone[:3]}"
print("every clip every run needs is present")

sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_config_guard.py "
   "tests/test_augment.py tests/test_layer_mix_and_sampler.py -q")

## 3. Run every block, then the summary and the download link

In [ ]:
import statistics as st
import zipfile

DEV_LINE = re.compile(r"best dev EER = ([0-9.]+)")
RUNS = []           # one row per run, saved to runs.json after each
CHOSEN = None       # the dev-selected candidate, once best_variant_seeds has run
DONE = False        # True once every block has finished


def score(tag, ckpt, manifest, encoder="wav2vec2-base"):
    args = [sys.executable, "-m", "src.training.evaluate", "--checkpoint", ckpt, "--manifest", manifest,
            "--encoder", encoder, "--data-root", ROOT, *SCORE_ARGS.split(),
            "--scores-out", OUT / f"{tag}_scores.csv", "--out", OUT / f"{tag}.json"]
    if SMOKE:
        args += ["--limit", "200"]
    code_, text = run_logged(args)
    try:
        pooled = json.loads((OUT / f"{tag}.json").read_text())["pooled"] if code_ == 0 else None
    except Exception:
        pooled = None
    if not pooled or "eer" not in pooled:
        failed(f"scoring {tag} (exit {code_})", tail(text))
        return None
    return pooled["eer"]


def pick_variant():
    """The pre-registered rule: lowest dev EER; ties within TIE go to fewer options."""
    pool = [r for r in PRIOR + RUNS if r["status"] == "ok" and r["dev_eer"] is not None
            and (r["block"] == "variants" or (r["block"] == "deployed_seeds" and r["seed"] == 1234))]
    if not pool:
        return None
    best = min(r["dev_eer"] for r in pool)
    tied = [r for r in pool if r["dev_eer"] <= best + TIE]
    return min(tied, key=lambda r: (len(r["options"]), r["dev_eer"]))


def published(prefix):
    out = {}
    for ev in EVALS:
        p = REPO / f"{prefix}__{ev}.json"
        if p.is_file():
            out[ev] = json.loads(p.read_text())["pooled"]["eer"]
    return out


def summary_text():
    """Health, every run's EERs, seed statistics and the variant choice, so far."""
    lines = ["STATUS: " + ("FINISHED - every block ran." if DONE else
                           f"IN PROGRESS - {len(RUNS)} run(s) finished so far."),
             "RUN HEALTH: " + ("every step ran without error." if not FAILURES else
                               f"{len(FAILURES)} step(s) FAILED -- see FAILURES.txt"),
             f"mode: {'SMOKE (numbers mean nothing)' if SMOKE else 'full'}   commit: {COMMIT}",
             "", "EER (%) per run",
             f"{'run':34s}{'status':>10s}{'dev':>8s}" + "".join(f"{ev:>11s}" for ev in EVALS)]
    for r in RUNS:
        cells = "".join(f"{'-' if r.get(ev) is None else format(100 * r[ev], '.2f'):>11s}" for ev in EVALS)
        dev = "-" if r["dev_eer"] is None else f"{100 * r['dev_eer']:.2f}"
        lines.append(f"{r['name']:34s}{r['status']:>10s}{dev:>8s}{cells}")

    def seed_table(system, block, title):
        rows = [r for r in RUNS if r["block"] == block and r["status"] == "ok"]
        values = {ev: [r[ev] for r in rows] for ev in EVALS}
        if block == "explore_seeds":  # the variant's own seed-1234 run, from an earlier run
            pub = {}
            for r in PRIOR:
                if r["status"] == "ok" and r.get("options") == VARIANTS.get(EXPLORE):
                    for ev in EVALS:
                        values[ev] = [r[ev]] + values[ev]
        else:
            pub = published(BASE[system][1])
            if block != "deployed_seeds":  # the published seed-1234 run joins the new seeds
                for ev in EVALS:
                    if ev in pub:
                        values[ev] = [pub[ev]] + values[ev]
        lines.append(f"\n{title}  ({len(values[next(iter(EVALS))])} seeds)")
        for ev in EVALS:
            v = [100 * x for x in values[ev]]
            ref = f"   published 1234: {100 * pub[ev]:.2f}" if ev in pub else ""
            if len(v) >= 2:
                lines.append(f"  {ev:9s} mean {st.mean(v):6.2f}  sd {st.stdev(v):5.2f}  "
                             f"min {min(v):6.2f}  max {max(v):6.2f}{ref}")
            elif v:
                lines.append(f"  {ev:9s} {v[0]:6.2f} (one run){ref}")

    if "deployed_seeds" in BLOCKS:
        seed_table("deployed", "deployed_seeds", "DEPLOYED S2 (published recipe), seeds incl. 1234 re-run")
    if "s3_seeds" in BLOCKS:
        seed_table("s3_xtts_channel", "s3_seeds", "S3 XTTS-only phone line, published 1234 + new seeds")
    if "xtts_channel_seeds" in BLOCKS:
        seed_table("xtts_channel", "xtts_channel_seeds", "S2 XTTS-only phone line, published 1234 + new seeds")
    if "explore_seeds" in BLOCKS:
        seed_table("deployed", "explore_seeds",
                   f"EXPLORATORY: {EXPLORE}, seed 1234 (earlier run) + new seeds, not a selection")
    if "variants" in BLOCKS:
        choice = pick_variant()
        lines.append("\nVARIANT SELECTION (dev EER only; test sets never used to choose)")
        for r in PRIOR + RUNS:
            if r["block"] == "variants" or (r["block"] == "deployed_seeds" and r["seed"] == 1234):
                dev = "-" if r["dev_eer"] is None else f"{100 * r['dev_eer']:.2f}"
                mark = "  <- chosen" if choice is not None and r["name"] == choice["name"] else ""
                earlier = "  (earlier run)" if r.get("earlier") else ""
                state = "" if r["status"] == "ok" else f"  ({r['status']}: not eligible)"
                lines.append(f"  {r['name']:34s} dev {dev}{mark}{earlier}{state}")
    return "\n".join(lines)


def write_summary():
    """summary.txt and summary.json as they stand now. Never stops the run."""
    try:
        text = summary_text()
    except Exception:
        text = "The summary could not be built:\n" + traceback.format_exc()
    try:
        (OUT / "summary.txt").write_text(text, encoding="utf-8")
        (OUT / "summary.json").write_text(json.dumps({
            "mode": ("smoke" if SMOKE else "full") if DONE else "in progress",
            "commit": COMMIT, "branch": BRANCH, "blocks": BLOCKS, "seeds": SEEDS,
            "variants": VARIANTS, "tie": TIE, "explore": EXPLORE,
            "chosen_variant": None if CHOSEN is None else CHOSEN["name"],
            "runs": RUNS, "failures": FAILURES}, indent=2, default=str))
    except Exception as e:
        print("(could not write the summary:", e, ")", flush=True)
    return text


def show_download_link():
    """Check the zip and show a link that downloads it."""
    from IPython.display import FileLink, display
    zip_path = WORK / "explore_seeds_results.zip"
    if not zip_path.is_file():
        print("No results zip was written -- see the errors above.")
        return
    with zipfile.ZipFile(zip_path) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"\nexplore_seeds_results.zip: {count} files, {zip_path.stat().st_size / 1e6:.1f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken} -- run section 4")
    os.chdir(WORK)  # the link is relative to the working directory
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink("explore_seeds_results.zip"))
    print("\nIf clicking does not download it: right-hand panel -> Output -> /kaggle/working ->",
          "explore_seeds_results.zip -> the three dots -> Download.")
    print("Download it BEFORE closing this tab: the session's files are deleted when it ends.")


def run_job(name, block, system, seed, options=None):
    """Train one config, score it on the three test sets, record and save the row."""
    t0 = time.time()
    row = {"name": name, "block": block, "system": system, "seed": seed,
           "options": options or {}, "dev_eer": None, "status": "failed"}
    try:
        base_config = BASE[system][0]
        cfg_path, cfg = make_config(name, base_config, seed, options)
        guard = sh(f"{sys.executable} -m src.training.config_guard {cfg_path}", check=False)
        if guard.returncode:
            failed(f"config_guard refused {name}", (guard.stdout + guard.stderr)[-2500:])
            return row
        code_, text = run_logged([sys.executable, "-m", "src.training.train", "--config", cfg_path,
                                  "--data-root", ROOT, "--device", "cuda"]
                                 + (["--smoke"] if SMOKE else []))
        ckpt = REPO / cfg["out_dir"] / "best.pt"
        if code_ and "model collapsed" in text:
            row["status"] = "collapsed"
            if not ckpt.is_file():
                return row
        elif code_:
            failed(f"training {name} (exit {code_})", tail(text))
            return row
        dev = DEV_LINE.findall(text)
        row["dev_eer"] = float(dev[-1]) if dev else None
        for ev, manifest in EVALS.items():
            row[ev] = score(f"{name}__{ev}", ckpt, manifest)
        if row["status"] != "collapsed":
            row["status"] = "ok" if all(row.get(ev) is not None for ev in EVALS) else "partial"
        (REPO / cfg["out_dir"] / "last.pt").unlink(missing_ok=True)  # resume file, not needed
    except Exception:
        failed(f"{name}: unexpected error", traceback.format_exc())
    finally:
        row["minutes"] = round((time.time() - t0) / 60, 1)
        RUNS.append(row)
        (OUT / "runs.json").write_text(json.dumps(RUNS, indent=2))
        write_summary()
        save_progress()
        shown = "  ".join(f"{ev} {100 * row[ev]:.2f}" for ev in EVALS if row.get(ev) is not None)
        dev_text = "-" if row["dev_eer"] is None else f"{100 * row['dev_eer']:.2f}"
        print(f"\n[{name}] {row['status']}  dev {dev_text}  {shown}  ({row['minutes']} min)\n",
              flush=True)
    return row


seeds = SEEDS[:1] if SMOKE else SEEDS
try:
    for block in BLOCKS:
        print(f"\n==================== {block} ====================\n", flush=True)
        if block == "deployed_seeds":
            for seed in [1234] + seeds:
                run_job(f"deployed_s{seed}", block, "deployed", seed)
        elif block == "variants":
            for vname, options in VARIANTS.items():
                run_job(f"deployed_{vname}_s1234", block, "deployed", 1234, options)
        elif block == "s3_seeds":
            for seed in seeds:
                run_job(f"s3xtts_s{seed}", block, "s3_xtts_channel", seed)
        elif block == "xtts_channel_seeds":
            for seed in seeds:
                run_job(f"xttsch_s{seed}", block, "xtts_channel", seed)
        elif block == "best_variant_seeds":
            CHOSEN = pick_variant()
            if CHOSEN is None:
                print("No completed published-recipe or variant run to choose from; skipped.")
            elif not CHOSEN["options"]:
                print("The published recipe has the lowest dev EER: no variant improves on it on dev,"
                      " so there is no variant to repeat. (This is itself the result.)")
            else:
                print(f"Dev-selected variant: {CHOSEN['name']} (dev EER {100 * CHOSEN['dev_eer']:.2f})")
                for seed in seeds[:2]:
                    run_job(f"{CHOSEN['name'].rsplit('_s', 1)[0]}_s{seed}", block, "deployed", seed,
                            CHOSEN["options"])
        elif block == "explore_seeds":
            if EXPLORE not in VARIANTS:
                failed(f"EXPLORE={EXPLORE!r} is not one of VARIANTS", "check the settings cell")
            else:
                for seed in seeds:
                    run_job(f"deployed_{EXPLORE}_s{seed}", block, "deployed", seed, VARIANTS[EXPLORE])
        else:
            failed(f"unknown block {block!r}", "check BLOCKS in the settings cell")
    DONE = True
    print("\nALL BLOCKS FINISHED\n", flush=True)
finally:  # even if something above broke: the summary, the zip and the link
    print(write_summary(), flush=True)
    save_progress()
    show_download_link()

## 4. Optional: the download link again

Section 3 already ends with the download link. Run this cell only if you need the
link again (for example after the page reconnects).

In [ ]:
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

WORKING = Path("/kaggle/working")
ZIP = WORKING / "explore_seeds_results.zip"
os.chdir(WORKING)
if not ZIP.is_file():
    print("No results zip yet -- section 3 writes it after its first finished run.")
else:
    with zipfile.ZipFile(ZIP) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"explore_seeds_results.zip: {count} files, {ZIP.stat().st_size / 1e6:.1f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken} -- run this cell again")
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink("explore_seeds_results.zip"))
    print("\nIf clicking does not download it: right-hand panel -> Output -> /kaggle/working ->",
          "explore_seeds_results.zip -> the three dots -> Download.")